In [1]:
import requests
from bs4 import BeautifulSoup
import time
import sqlite3

# データベースに接続
db_name = 'github_repos.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# テーブルの初期化
cursor.execute("DROP TABLE IF EXISTS google_repos")
cursor.execute("""
    CREATE TABLE google_repos (
        repo_name TEXT,
        language TEXT,
        stars INTEGER
    )
""")
conn.commit()
print("データベースとテーブルの準備が完了しました。")

データベースとテーブルの準備が完了しました。


In [2]:
url = 'https://github.com/google?tab=repositories'
print(f"アクセス中: {url}")

# スクレイピング実行
response = requests.get(url)

# ★重要: 課題要件の待機処理
time.sleep(1)

soup = BeautifulSoup(response.text, 'html.parser')

# リポジトリのリストを取得
repos = soup.select('div#user-repositories-list li')

print(f"{len(repos)} 件のリポジトリが見つかりました。")

# 取得したデータをループ処理
for repo in repos:
    try:
        # 1. リポジトリ名
        name_tag = repo.select_one('h3 a')
        name = name_tag.text.strip() if name_tag else "Unknown"

        # 2. 言語 
        lang_tag = repo.select_one('span[itemprop="programmingLanguage"]')
        language = lang_tag.text.strip() if lang_tag else "Unknown"

        # 3. スター数
        star_tag = repo.select_one('a[href*="/stargazers"]')
        if star_tag:
            star_text = star_tag.text.strip().replace(',', '')
            # 'k' 表記への対応 
            if 'k' in star_text.lower():
                stars = int(float(star_text.lower().replace('k', '')) * 1000)
            else:
                stars = int(star_text)
        else:
            stars = 0

        # DBへ保存
        cursor.execute("INSERT INTO google_repos VALUES (?, ?, ?)", (name, language, stars))
        
    except Exception as e:
        print(f"エラースキップ: {e}")

conn.commit()
print("データの保存が完了しました。")

アクセス中: https://github.com/google?tab=repositories
0 件のリポジトリが見つかりました。
データの保存が完了しました。


In [3]:
# スター数が多い順にトップ10を表示
cursor.execute("SELECT * FROM google_repos ORDER BY stars DESC LIMIT 10")
rows = cursor.fetchall()

print("--- 保存データ確認 (Top 10) ---")
print(f"{'Repository':<30} | {'Language':<15} | {'Stars'}")
print("-" * 60)
for row in rows:
    print(f"{row[0]:<30} | {row[1]:<15} | {row[2]}")

# 処理が終わったら接続を閉じる
conn.close()

--- 保存データ確認 (Top 10) ---
Repository                     | Language        | Stars
------------------------------------------------------------
